# Custom surface site analysis

Edit **Cell 2 only**, then **Run All**.


In [1]:

# Imports (DO NOT EDIT)
import sys
sys.path.append(".")

from ase.io import read, Trajectory
from acat.settings import CustomSurface
from acat.adsorption_sites import SlabAdsorptionSites

import site_analysis as sa


In [2]:

# USER INPUT (EDIT ONLY THIS CELL)
structure_file = "Ru_0001_vacancy.xyz"
n_layers = 4
adsorbate_height = 2.0
site_bond_cutoff = 2.5


In [3]:

# Load slab
slab = read(structure_file)
surface = CustomSurface(slab, n_layers=n_layers)
nslab = len(slab)


In [4]:

# Generate symmetry-unique site geometries
cas = SlabAdsorptionSites(slab, surface, composition_effect=True)

single_geoms, single_sites_lists = sa.generate_unique_sites(
    slab,
    cas.get_sites(),
    nslab,
    site_bond_cutoff,
    adsorbate_height
)

traj = Trajectory("unique_sites.traj", "w")
for g in single_geoms:
    traj.write(g)
traj.close()


In [5]:

# Extract 3-fold site graphs
admols, threefold_geom_indices = sa.classify_threefold_sites(
    single_geoms, single_sites_lists
)


In [6]:

# Graph isomorphism clustering
iso_mat, clusters = sa.cluster_isomorphic_graphs(admols)


In [7]:

# Update 3-fold site labels
type_map = sa.update_threefold_site_labels(
    single_sites_lists,
    clusters,
    threefold_geom_indices
)


In [8]:

# Write 3-fold-only trajectory
traj3 = Trajectory("threefold_sites.traj", "w")
for i in threefold_geom_indices:
    traj3.write(single_geoms[i])
traj3.close()


In [9]:

# Pairwise strict isomorphism (PRINT)
print("Pairwise strict isomorphism:")
for i in range(len(admols)):
    for j in range(i + 1, len(admols)):
        print(f"{i} vs {j} =", iso_mat[i, j])


Pairwise strict isomorphism:
0 vs 1 = False
0 vs 2 = False
0 vs 3 = False
0 vs 4 = False
1 vs 2 = False
1 vs 3 = False
1 vs 4 = False
2 vs 3 = False
2 vs 4 = False
3 vs 4 = False


In [10]:

# Distinct 3-fold site types (PRINT)
print("Number of distinct 3-fold site types:", len(clusters))
for members in clusters.values():
    print("3-fold site type:", members)


Number of distinct 3-fold site types: 5
3-fold site type: [0]
3-fold site type: [1]
3-fold site type: [2]
3-fold site type: [3]
3-fold site type: [4]


In [11]:

# Updated 3-fold site labels (PRINT)
print("Updated 3-fold site labels per geometry:")
for geom_idx, label in type_map.items():
    print(f"Geometry {geom_idx} -> {label}")


Updated 3-fold site labels per geometry:
Geometry 4 -> 3fold1
Geometry 5 -> 3fold2
Geometry 6 -> 3fold3
Geometry 11 -> 3fold4
Geometry 14 -> 3fold5


In [12]:
# Site yaml file generated
print("All sites for the custom surfaces are saved in site.yaml")
sa.write_sites_yaml(single_sites_lists, clusters)


All sites for the custom surfaces are saved in site.yaml
